# Governance Framework — Executed Analysis

This notebook runs directly against the live warehouse (`data/warehouse/governance.duckdb`) and every cell below produces real output from that run — nothing here is asserted without a query backing it.

**Read this first:** all data in this warehouse is synthetic, generated by `src/pipeline.py` with a fixed random seed, reseeded relative to the date the pipeline was last run. The patterns found below (DS-003, DS-012, the SLA inversion) are real behaviors of this specific pipeline run, and the analytical technique used to find them generalizes to real data — but the underlying data itself is not sourced from a real banking system. See `project_metrics.md` for the full limitations statement.

In [1]:
import duckdb
import pandas as pd

conn = duckdb.connect('../data/warehouse/governance.duckdb', read_only=True)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

## 1. Understanding the Data

In [2]:
conn.execute("""
    SELECT category, regulatory_criticality, COUNT(*) as dataset_count
    FROM data_inventory
    GROUP BY category, regulatory_criticality
    ORDER BY category
""").df()

,category,regulatory_criticality,dataset_count
0,Business Critical,Medium,4
1,Operational Critical,Low,3
2,Reference Data,Low,2
3,Regulatory Critical,High,4


In [3]:
conn.execute("""
    SELECT dataset_id, COUNT(*) as tests_run,
        SUM(CASE WHEN status='Fail' THEN 1 ELSE 0 END) as fails,
        ROUND(100.0*SUM(CASE WHEN status='Fail' THEN 1 ELSE 0 END)/COUNT(*),1) as fail_rate_pct
    FROM control_test_results
    GROUP BY dataset_id
    ORDER BY fail_rate_pct DESC
""").df()

,dataset_id,tests_run,fails,fail_rate_pct
0,DS-003,150,150.0,100.0
1,DS-010,90,90.0,100.0
2,DS-009,60,60.0,100.0
3,DS-006,90,90.0,100.0
4,DS-002,180,180.0,100.0
5,DS-007,90,89.0,98.9
6,DS-005,90,81.0,90.0
7,DS-008,60,46.0,76.7
8,DS-001,240,183.0,76.3
9,DS-011,60,28.0,46.7


In [4]:
conn.execute("""
    SELECT status, COUNT(*) as n, ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (),1) as pct
    FROM control_test_results GROUP BY status
""").df()

,status,n,pct
0,Fail,1078,79.9
1,Pass,272,20.1


## 2. Finding Patterns

Four pattern queries, run directly against live data.

### 2a. Trust score vs. watchlist status by domain

In [5]:
conn.execute("""
    SELECT dts.domain, dts.trust_score, dts.trust_category,
        count(*) filter (where dw.watchlist_status='Watchlist') as n_watchlist,
        count(*) filter (where dw.watchlist_status='Monitor') as n_monitor
    FROM domain_trust_scores dts
    LEFT JOIN data_inventory di ON di.domain = dts.domain
    LEFT JOIN dq_watchlist dw ON dw.dataset_id = di.dataset_id
    GROUP BY dts.domain, dts.trust_score, dts.trust_category
    ORDER BY dts.trust_score DESC
""").df()

,domain,trust_score,trust_category,n_watchlist,n_monitor
0,Finance,98.9,Trusted,0,1
1,Product,96.9,Trusted,0,1
2,HR,95.7,Trusted,1,0
3,Risk,94.1,Trusted,0,1
4,Customer,93.8,Trusted,0,2
5,Reference,92.3,Trusted,2,0
6,Enterprise (Overall),90.6,Trusted,0,0
7,Transaction,83.0,Monitor,1,0
8,Operations,81.0,Monitor,3,0
9,Sales,80.0,Monitor,1,0


### 2b. Control failure rate by category

In [6]:
conn.execute("""
    SELECT cr.category, COUNT(*) as total_tests,
        SUM(CASE WHEN ctr.status='Fail' THEN 1 ELSE 0 END) as failures,
        ROUND(100.0*SUM(CASE WHEN ctr.status='Fail' THEN 1 ELSE 0 END)/COUNT(*),1) as fail_rate
    FROM control_test_results ctr
    JOIN control_rulebook cr ON ctr.rule_id = cr.rule_id
    GROUP BY cr.category ORDER BY fail_rate DESC
""").df()

,category,total_tests,failures,fail_rate
0,Consistency,330,291.0,88.2
1,Accuracy,270,225.0,83.3
2,Completeness,420,343.0,81.7
3,Timeliness,330,219.0,66.4


### 2c. SLA breach rate by severity — the headline secondary pattern

In [7]:
conn.execute("""
    SELECT severity, COUNT(*) as total,
        SUM(CASE WHEN sla_breach THEN 1 ELSE 0 END) as breached,
        ROUND(100.0*SUM(CASE WHEN sla_breach THEN 1 ELSE 0 END)/COUNT(*),1) as breach_rate,
        SUM(CASE WHEN status='Resolved' THEN 1 ELSE 0 END) as ever_resolved,
        ROUND(100.0*SUM(CASE WHEN status='Resolved' THEN 1 ELSE 0 END)/COUNT(*),1) as pct_ever_resolved
    FROM remediation_tickets
    GROUP BY severity ORDER BY breach_rate DESC
""").df()

,severity,total,breached,breach_rate,ever_resolved,pct_ever_resolved
0,High,15,11.0,73.3,2.0,13.3
1,Critical,19,10.0,52.6,6.0,31.6
2,Medium,8,0.0,0.0,2.0,25.0


**Counterintuitive result:** High-severity tickets (3-day SLA) breach at 73.3%, *higher* than Critical (1-day SLA) at 52.6%. The `pct_ever_resolved` column shows why: only 13.3% of High tickets have ever been resolved at all, vs. 31.6% of Critical — the gap isn't speed (both average 1.0 day to resolve once someone starts), it's that High tickets get worked far less often in the first place.

### 2d. Watchlist priority vs. actual failure volume

In [8]:
conn.execute("""
    SELECT dw.dataset_id, dw.watchlist_status, dw.priority,
        COUNT(ctr.test_id) as total_tests,
        SUM(CASE WHEN ctr.status='Fail' THEN 1 ELSE 0 END) as fails
    FROM dq_watchlist dw
    JOIN control_test_results ctr ON ctr.dataset_id = dw.dataset_id
    GROUP BY dw.dataset_id, dw.watchlist_status, dw.priority
    ORDER BY fails DESC
""").df()

,dataset_id,watchlist_status,priority,total_tests,fails
0,DS-001,Monitor,Medium,240,183.0
1,DS-002,Watchlist,High,180,180.0
2,DS-003,Monitor,Medium,150,150.0
3,DS-010,Watchlist,Medium,90,90.0
4,DS-006,Watchlist,Medium,90,90.0
5,DS-007,Monitor,Low,90,89.0
6,DS-005,Monitor,Low,90,81.0
7,DS-009,Watchlist,Medium,60,60.0
8,DS-008,Watchlist,Medium,60,46.0
9,DS-004,Monitor,Medium,120,34.0


## 3. The Headline Finding — DS-003: 100% Failure, Still "Trusted"

Risk Exposure (DS-003) failed **every one of its 150 control tests** over the full 30-day window. Its domain (Risk) is nonetheless scored 94.1/100, "Trusted" — a direct contradiction worth digging into.

In [9]:
conn.execute("""
    SELECT dataset_id, COUNT(*) as total_tests,
        SUM(CASE WHEN status='Fail' THEN 1 ELSE 0 END) as fails,
        ROUND(100.0*SUM(CASE WHEN status='Fail' THEN 1 ELSE 0 END)/COUNT(*),1) as fail_rate,
        ROUND(AVG(control_effectiveness),1) as avg_effectiveness
    FROM control_test_results
    WHERE dataset_id = 'DS-003'
    GROUP BY dataset_id
""").df()

,dataset_id,total_tests,fails,fail_rate,avg_effectiveness
0,DS-003,150,150.0,100.0,95.0


In [10]:
conn.execute("""
    SELECT ctr.rule_id, cr.rule_name, cr.threshold, cr.severity,
        ROUND(AVG(ctr.control_effectiveness),2) as avg_eff, COUNT(*) as n_tests,
        SUM(CASE WHEN ctr.status='Fail' THEN 1 ELSE 0 END) as fails
    FROM control_test_results ctr
    JOIN control_rulebook cr ON cr.rule_id = ctr.rule_id
    WHERE ctr.dataset_id = 'DS-003'
    GROUP BY ctr.rule_id, cr.rule_name, cr.threshold, cr.severity
""").df()

,rule_id,rule_name,threshold,severity,avg_eff,n_tests,fails
0,COMP-004,Risk Rating Populated,98.0,High,94.98,30,30.0
1,TIME-003,Risk Exposure Daily Refresh,99.5,Critical,95.00,30,30.0
2,ACCU-004,Risk Score Valid Range,99.5,High,94.84,30,30.0
3,CONS-005,Risk Tier Consistency,98.0,High,95.07,30,30.0
4,TIME-007,Risk Report Delivery SLA,100.0,Critical,94.94,30,30.0


**Why this happens:** every one of DS-003's 5 controls averages 94.8–95.1% effectiveness — high in absolute terms, but just under each rule's 98–100% threshold, so every single test is classified `Fail`. The Data Trust Score formula averages continuous `control_effectiveness`, not binary pass/fail outcomes, and DS-003 is only "Monitor" status (1-point penalty) rather than "Watchlist," so the 100% fail rate barely dents the score. A dataset can fail literally every test it runs and still be labeled "Trusted."

**Proposed fix (demonstrated in Section 6 below):** a binary floor — any dataset with a 100% fail rate in its trailing 7-day window is capped at "At Risk" regardless of average effectiveness.

## 4. Where the Framework Is Too Strict — DS-012 False Positive

The opposite failure mode: a near-zero absolute change classified as a severe alert.

In [11]:
# The stored *_early columns are rounded to 2dp for display. The pipeline's own watchlist_reason
# is computed from the UNROUNDED value - so read the reason text, don't recompute from the column.
conn.execute("""
    SELECT dataset_id, duplicate_rate_early AS early_rounded_for_display,
           duplicate_rate_recent, duplicate_rate_trend, watchlist_reason
    FROM dq_watchlist WHERE dataset_id = 'DS-012'
""").df()

,dataset_id,early_rounded_for_display,duplicate_rate_recent,duplicate_rate_trend,watchlist_reason
0,DS-012,0.01,0.14,Deteriorating,Null rate increased 16.2%; Duplicate rate incr...


In [12]:
# Recover the TRUE unrounded baseline from the pipeline's own reported percentage,
# and check whether the max(x, 0.01) division-guard floor actually engaged.
import re
ds, recent, reason = conn.execute("""
    SELECT dataset_id, duplicate_rate_recent, watchlist_reason
    FROM dq_watchlist WHERE dataset_id='DS-012'
""").fetchone()
pct = float(re.search(r'Duplicate rate increased ([\d.]+)%', reason).group(1))
true_early = recent / (1 + pct/100)
print(f"reported by pipeline : {pct}%")
print(f"recomputed from the ROUNDED column: {((recent-0.01)/max(0.01,0.01))*100:.1f}%   <-- wrong by ~107pp")
print(f"true unrounded baseline           : {true_early:.5f}")
print(f"absolute move                     : {recent-true_early:.3f} percentage points")
print(f"did the max(x,0.01) floor engage? : {true_early < 0.01}")

reported by pipeline : 1192.9%
recomputed from the ROUNDED column: 1300.0%   <-- wrong by ~107pp
true unrounded baseline           : 0.01083
absolute move                     : 0.129 percentage points
did the max(x,0.01) floor engage? : False


In [13]:
# Does that floor engage anywhere at all, on any of the three signals?
conn.execute("""
    SELECT MIN(duplicate_rate_early) AS min_dup_early,
           MIN(null_rate_early) AS min_null_early,
           MIN(control_failure_rate_early) AS min_ctrl_early
    FROM dq_watchlist
""").df()

,min_dup_early,min_null_early,min_ctrl_early
0,0.01,0.25,0.67


A **0.129 percentage-point** move (0.0108% → 0.14%) on a monthly-refreshed dataset is reported as a **1192.9% increase** — the largest single movement on the entire watchlist. DS-003 (Section 3) shows the framework too lenient (total failure, still "Trusted"); DS-012 shows it too strict (a non-event, flagged loudest). Same class of formula problem, opposite ends.

Two corrections worth recording, since the first pass at this got both wrong:

1. **1,192.9%, not 1,300%.** Recomputing from the stored `duplicate_rate_early` column gives 1,300% — but that column is rounded to 2dp. The pipeline's own `watchlist_reason`, computed from the unrounded value, says 1,192.9%. Deriving a metric from a rounded intermediate rather than from source produced an answer ~107 percentage points off that silently disagreed with the project's own `COMPREHENSIVE_PROJECT_REPORT.md`.
2. **The `max(x, 0.01)` floor never fired.** The true baseline (0.01083) is *above* it — and per the cell above, the minimum early value across all three signals and all 13 datasets never drops below 0.01, so the floor doesn't engage anywhere in this data. The cause is percentage-change-on-a-near-zero-baseline generally; the floor is a latent version of the same hazard, not the active one.

## 5. Degenerate Metric Proof

In [14]:
conn.execute("""
    SELECT dataset_id, COUNT(DISTINCT rule_id) as controls_assigned,
        ROUND(100.0*COUNT(DISTINCT rule_id)/30,1) as pct_of_30_rule_catalog
    FROM control_test_results GROUP BY dataset_id ORDER BY pct_of_30_rule_catalog DESC
""").df()

,dataset_id,controls_assigned,pct_of_30_rule_catalog
0,DS-001,8,26.7
1,DS-002,6,20.0
2,DS-003,5,16.7
3,DS-004,4,13.3
4,DS-005,3,10.0
5,DS-007,3,10.0
6,DS-010,3,10.0
7,DS-006,3,10.0
8,DS-012,2,6.7
9,DS-008,2,6.7


In [15]:
conn.execute("SELECT * FROM governance_maturity").df()

,maturity_score,maturity_level,control_coverage,automation_pct,sla_compliance,audit_completeness
0,83.0,Managed,100.0,85.0,50.0,100.0


Per-dataset control coverage actually *varies* (6.7%–26.7% of the 30-rule catalog) — that is not the degenerate metric. The genuinely degenerate value is `governance_maturity.control_coverage`, a single enterprise-level constant (always 100.0, since all 13 datasets have at least one control assigned) alongside `audit_completeness` (hardcoded 100.0). Two of the four weighted inputs to Governance Maturity cannot move regardless of what actually happens operationally.

## 6. What-If Scenarios

### 6a. Enterprise trust score with watchlist/monitor penalty removed

The enterprise score is the *mean of the per-domain scores*, and each domain score is
`avg_control_effectiveness - 3*watchlist_count - 1*monitor_count`. So "what if every watchlist and
monitor flag were cleared?" = the mean of the per-domain `avg_control_effectiveness`, with no penalty
subtracted. Computing it that way (rather than as a flat average over raw test rows) is what makes it
directly comparable to the stored enterprise score.

In [16]:
conn.execute("""
    SELECT ROUND(AVG(avg_control_effectiveness),1) AS enterprise_score_if_no_penalty
    FROM domain_trust_scores
    WHERE domain <> 'Enterprise (Overall)'
""").df()

,enterprise_score_if_no_penalty
0,93.9


In [17]:
conn.execute("SELECT domain, trust_score FROM domain_trust_scores WHERE domain='Enterprise (Overall)'").df()

,domain,trust_score
0,Enterprise (Overall),90.6


Current enterprise score 90.6 vs. 93.9 with the watchlist/monitor penalty zeroed out — a real but modest 3.3-point ceiling on what resolving every open issue could buy today (both endpoints remain "Trusted").

### 6b. Demonstrate the DS-003 fix — binary floor

In [18]:
conn.execute("""
    SELECT dts.domain, dts.trust_score as current_score, dts.trust_category as current_category,
        fr.total_fail_rate,
        CASE WHEN fr.total_fail_rate = 1.0 THEN 0 ELSE dts.trust_score END as fixed_score,
        CASE WHEN fr.total_fail_rate = 1.0 THEN 'At Risk' ELSE dts.trust_category END as fixed_category
    FROM domain_trust_scores dts
    JOIN data_inventory di ON di.domain = dts.domain
    JOIN (
        SELECT dataset_id, ROUND(1.0*SUM(CASE WHEN status='Fail' THEN 1 ELSE 0 END)/COUNT(*),3) as total_fail_rate
        FROM control_test_results GROUP BY dataset_id
    ) fr ON fr.dataset_id = di.dataset_id
    WHERE dts.domain = 'Risk'
""").df()

,domain,current_score,current_category,total_fail_rate,fixed_score,fixed_category
0,Risk,94.1,Trusted,1.0,0.0,At Risk


Under the binary floor, Risk domain's score flips from **94.1 "Trusted" → 0.0 "At Risk"** — an executed result, not a hypothetical.

### 6c. Demonstrate the DS-003 fix — failure-streak escalation

In [19]:
conn.execute("""
    WITH chronic AS (
        SELECT ctr.dataset_id, ctr.rule_id, cr.severity, cr.rule_name,
            COUNT(*) as failure_count
        FROM control_test_results ctr
        JOIN control_rulebook cr ON cr.rule_id = ctr.rule_id
        WHERE ctr.status='Fail'
        GROUP BY ctr.dataset_id, ctr.rule_id, cr.severity, cr.rule_name
        HAVING COUNT(*) >= 27
    )
    SELECT dataset_id, rule_id, rule_name, failure_count, severity as current_severity,
        CASE severity
            WHEN 'Low' THEN 'Medium'
            WHEN 'Medium' THEN 'High'
            WHEN 'High' THEN 'Critical'
            ELSE 'Critical'
        END as escalated_severity
    FROM chronic
    ORDER BY failure_count DESC, dataset_id
""").df()

,dataset_id,rule_id,rule_name,failure_count,current_severity,escalated_severity
0,DS-001,COMP-001,Customer ID Not Null,30,Critical,Critical
1,DS-001,CONS-007,Customer Status Consistency,30,Medium,High
2,DS-001,CONS-001,Customer-Account Referential Integrity,30,Critical,Critical
3,DS-002,ACCU-002,Date Range Validity,30,High,Critical
4,DS-002,TIME-002,Transaction Feed SLA,30,Critical,Critical
5,DS-002,COMP-002,Transaction Amount Not Null,30,Critical,Critical
6,DS-002,ACCU-001,Transaction Amount Positive,30,Critical,Critical
7,DS-002,CONS-002,Transaction-Account Consistency,30,Critical,Critical
8,DS-002,CONS-006,GL-Transaction Reconciliation,30,Critical,Critical
9,DS-003,TIME-003,Risk Exposure Daily Refresh,30,Critical,Critical


33 dataset-rule pairs failed 27-30 of the last 30 days. Under a "7+ consecutive days → escalate one severity tier" rule, most Medium/High rules above would jump to High/Critical — currently every one of these generates the same single ticket regardless of how long the failure has persisted (see `sql/v2_rebuild/SQL_COMPARISON.md` Edge Case 4).

### 6d. Trust score formula vs. a simpler alternative

In [20]:
conn.execute("""
    WITH bounds AS (SELECT max(test_date) as max_date FROM control_test_results),
    recent AS (
        SELECT r.*, di.domain FROM control_test_results r
        JOIN data_inventory di ON di.dataset_id = r.dataset_id
        -- >= matches v1's inclusive 7-day window; see sql/v2_rebuild/SQL_COMPARISON.md file 03
        WHERE r.test_date >= (SELECT max_date FROM bounds) - INTERVAL 7 DAY
    ),
    simple AS (
        SELECT domain, ROUND(100.0*SUM(CASE WHEN status='Pass' THEN 1 ELSE 0 END)/COUNT(*),1) as simple_compliance_rate
        FROM recent GROUP BY domain
    )
    SELECT dts.domain, dts.trust_score as current_formula_score, dts.trust_category as current_category,
        s.simple_compliance_rate
    FROM domain_trust_scores dts
    LEFT JOIN simple s ON s.domain = dts.domain
    ORDER BY dts.trust_score DESC
""").df()

,domain,current_formula_score,current_category,simple_compliance_rate
0,Finance,98.9,Trusted,90.6
1,Product,96.9,Trusted,0.0
2,HR,95.7,Trusted,62.5
3,Risk,94.1,Trusted,0.0
4,Customer,93.8,Trusted,13.6
5,Reference,92.3,Trusted,59.4
6,Enterprise (Overall),90.6,Trusted,NaN
7,Transaction,83.0,Monitor,0.0
8,Operations,81.0,Monitor,8.9
9,Sales,80.0,Monitor,0.0


The divergence is stark for some domains: Risk (94.1 "Trusted") and Product (96.9 "Trusted") both show **0.0%** simple pass/fail compliance; Customer (93.8 "Trusted") shows 13.6%. Other domains hold up better under both views (Finance: 98.9 vs. 90.6%). Neither formula is objectively correct — the current weighted formula is more forgiving of near-miss thresholds and factors in early-warning trend data the simple version ignores entirely, but that forgiveness is exactly what let DS-003 score "Trusted." The simple version is more alarming but throws away the trend signal that makes the Watchlist layer valuable. This is a real tradeoff, not a case of one formula being wrong.

## 7. One-Layer Counterfactual — What If the Watchlist (Layer 5) Didn't Exist?

In [21]:
# Would DS-003's problem still be visible using only control_test_results (Layer 3), no dq_watchlist?
conn.execute("""
    SELECT dataset_id, COUNT(*) as tests, SUM(CASE WHEN status='Fail' THEN 1 ELSE 0 END) as fails
    FROM control_test_results WHERE dataset_id='DS-003' GROUP BY dataset_id
""").df()

,dataset_id,tests,fails
0,DS-003,150,150.0


In [22]:
# Would DS-012's false-positive trend signal exist without dq_watchlist?
# Layer 3's table has no null-rate or duplicate-rate concept at all - list its columns to prove it.
cols = conn.execute("DESCRIBE control_test_results").df()['column_name'].tolist()
print("control_test_results columns:", cols)
print("any null-rate column? ", [c for c in cols if 'null' in c.lower()])
print("any duplicate-rate column?", [c for c in cols if 'dup' in c.lower()])

control_test_results columns: ['test_id', 'test_date', 'dataset_id', 'rule_id', 'total_records', 'pass_count', 'fail_count', 'failure_rate', 'control_effectiveness', 'status']
any null-rate column?  []
any duplicate-rate column? []


**Result:** DS-003's 100%-fail problem is directly visible in `control_test_results` alone (150/150 fails) — Layer 3 (Detect) catches it without needing Layer 5 at all. But DS-012's false positive is a `dq_watchlist`-only concept: `control_test_results` has no null-rate or duplicate-rate columns whatsoever, so that entire signal — and the entire "too strict" failure mode found in Section 4 — would not exist without Layer 5. This distinguishes what the Watchlist layer uniquely contributes (trend detection on dimensions Layer 3 doesn't test at all) from what's redundant with Detect (raw failure visibility, which Layer 3 already provides).

## 8. Named Limitations (In Code)

What this notebook's analysis can and cannot confirm, demonstrated directly rather than just asserted in prose.

In [23]:
conn.execute("SELECT COUNT(*) as watchlist_row_count FROM dq_watchlist").df()

,watchlist_row_count
0,13


`dq_watchlist` has exactly **13 rows — one per dataset, a point-in-time snapshot**, not a dated daily history. This is direct proof (not an assertion) that no query against this warehouse can backtest a claim like "flagged N days before failure" — there is no historical watchlist-status-by-date to diff against a later exception date.

**What this analysis cannot confirm:**
- A specific backtested early-warning lead time (the data structurally doesn't support it — see above).
- A real false-positive rate for the watchlist (no ground-truth "this alert was wrong" label exists in this warehouse).
- Any day-of-week SLA calendar effect (checked in `SQL_COMPARISON.md` Edge Case 3 — sample sizes too small to be conclusive).
- That any of these findings describe a real banking data quality problem. Every number in this notebook comes from `src/pipeline.py`'s synthetic data generator, seeded to the run date. The patterns found (DS-003, DS-012, the SLA inversion) are real behaviors of this specific framework and its scoring formulas — genuine design-methodology findings — but not discoveries about real production data.